In [6]:
import re
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

STATIONS = r"C:\Users\kayle\Downloads\stations.csv"
CITY_FILES = [r"C:\Users\kayle\Downloads\cities_us.csv",
              r"C:\Users\kayle\Downloads\cities_sa_au.csv"]

def parse_point(s):
    nums = re.findall(r"-?\d+(?:\.\d+)?", str(s))
    return (float(nums[0]), float(nums[1])) if len(nums) >= 2 else (np.nan, np.nan)

def clean(s):
    s = str(s).strip().strip("<>")
    return re.sub(r"@en$", "", s).strip('"')

cities = pd.concat([pd.read_csv(f) for f in CITY_FILES], ignore_index=True)
cities.columns = [c.lstrip("?") for c in cities.columns]
cities[["lon", "lat"]] = cities["coord"].apply(lambda s: pd.Series(parse_point(s)))
cities["city_name"] = cities["cityLabel"].map(clean)
cities["city_qid"] = cities["city"].map(clean).str.split("/").str[-1]
cities = cities.dropna(subset=["lat", "lon"]).drop_duplicates("city_qid").reset_index(drop=True)

stations = pd.read_csv(STATIONS).dropna(subset=["LAT", "LON"]).reset_index(drop=True)

tree = BallTree(np.radians(cities[["lat", "lon"]].to_numpy()), metric="haversine")
dist, idx = tree.query(np.radians(stations[["LAT", "LON"]].to_numpy()), k=1)

nearest = cities.iloc[idx[:, 0]].reset_index(drop=True)
stations["nearest_city"] = nearest["city_name"]
stations["city_qid"] = nearest["city_qid"]
stations["distance_km"] = (dist[:, 0] * 6371).round(1)

stations.to_csv("station_cities.csv", index=False)

print(len(cities), "cities loaded")
print(stations.head(10))
print(stations["distance_km"].describe())

32794 cities loaded
    STATION_ID   LAT     LON nearest_city city_qid  distance_km
0  ASM00094275 -18.3  143.55   Georgetown  Q182262          0.0
1  ASM00094275 -18.3  143.55   Georgetown  Q182262          0.0
2  ASM00094275 -18.3  143.55   Georgetown  Q182262          0.0
3  ASM00094275 -18.3  143.55   Georgetown  Q182262          0.0
4  ASM00094275 -18.3  143.55   Georgetown  Q182262          0.0
5  ASM00094275 -18.3  143.55   Georgetown  Q182262          0.0
6  ASM00094275 -18.3  143.55   Georgetown  Q182262          0.0
7  ASM00094275 -18.3  143.55   Georgetown  Q182262          0.0
8  ASM00094275 -18.3  143.55   Georgetown  Q182262          0.0
9  ASM00094275 -18.3  143.55   Georgetown  Q182262          0.0
count    558396.000000
mean         22.257364
std          40.246457
min           0.000000
25%           2.500000
50%           9.600000
75%          31.600000
max        1742.600000
Name: distance_km, dtype: float64


In [9]:
stations = pd.read_csv(STATIONS).dropna(subset=["LAT", "LON"]).reset_index(drop=True)

In [10]:
stations = pd.read_csv(STATIONS).dropna(subset=["LAT", "LON"]).drop_duplicates("STATION_ID").reset_index(drop=True)

In [12]:
dist, idx = tree.query(np.radians(stations[["LAT", "LON"]].to_numpy()), k=1)
nearest = cities.iloc[idx[:, 0]].reset_index(drop=True)
stations["nearest_city"] = nearest["city_name"]
stations["city_qid"] = nearest["city_qid"]
stations["distance_km"] = (dist[:, 0] * 6371).round(1)

stations.to_csv(r"C:\Users\kayle\Downloads\station_cities.csv", index=False)
print(len(stations))
stations.head()

13288


,STATION_ID,LAT,LON,nearest_city,city_qid,distance_km
0,ASM00094275,-18.30,143.55,Georgetown,Q182262,0.0
1,ASM00094299,-16.29,149.96,Flying Fish Point,Q21911209,433.8
2,ASM00094940,-35.08,150.80,Vincentia,Q3559914,11.5
3,ASM00094995,-31.54,159.08,Arakoon,Q4783860,575.1
4,ASM00094997,-53.10,73.72,Arrabales de la ciudad islámica,Q97621954,878.6
